In [ ]:
import torch,torchvision,sklearn,albumentations,os
import pandas as pd
import cv2
from torch.utils.data import Dataset
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

In [ ]:
base_image_dir = './data/train/train_images/'

df = pd.read_csv('./data/train/train.csv')


# create a new column 'filename' that matches actual file on disk.
df['filename'] = df['id_code'].astype(str) + '.png'

#reduce the number of classes from 5 to 3 for better class balance
def reduce_classes(label):
    if label == 0: return 0        # Normal
    elif label <= 2: return 1      # Mild/Moderate
    else: return 2                 # Severe/Proliferative
df['diagnosis_3cat'] = df['diagnosis'].apply(reduce_classes)


print(df.head())

        id_code  diagnosis          filename  diagnosis_3cat
0  000c1434d8d7          2  000c1434d8d7.png               1
1  001639a390f0          4  001639a390f0.png               2
2  0024cdab0c1e          1  0024cdab0c1e.png               1
3  002c21358ce6          0  002c21358ce6.png               0
4  005b95c28852          0  005b95c28852.png               0


In [ ]:

class RetinopathyDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Get the filename from the dataframe at row 'idx'
        img_name = os.path.join(self.root_dir, self.df.iloc[idx]['filename'])
        
        # 2. Load the image
        image = Image.open(img_name)
        
        # 3. Get the label (0, 1, or 2)
        label = int(self.df.iloc[idx]['diagnosis_3cat'])

        if self.transform:
            image = self.transform(image)

        return image, label